In [2]:
# Import foundational libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import for preprocessing and splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Import the models
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Import for evaluation
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score

# Import for saving the model
import joblib

# Set a style for our plots
sns.set_style('whitegrid')

In [3]:
# Load the dataset
# Make sure your 'data.csv' file is in the same folder as your notebook
df = pd.read_csv('data.csv')

# Display the first 5 rows to see what it looks like
print("--- First 5 Rows ---")
df.head()

--- First 5 Rows ---


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
# 'id' is just an identifier. We don't need it.
df = df.drop(['id'], axis=1)

# Check the first 5 rows again to confirm it's gone
print("\n--- After Dropping 'id' Column ---")
df.head()


--- After Dropping 'id' Column ---


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
# See how many Malignant vs. Benign cases we have
print("\n--- Target Distribution ---")
print(df['diagnosis'].value_counts())

# Use LabelEncoder to turn 'M' into 1 and 'B' into 0
encoder = LabelEncoder()
df['diagnosis'] = encoder.fit_transform(df['diagnosis'])

# Now 'diagnosis' is 1 or 0
print("\n--- After Encoding Target ---")
print(df.head())


--- Target Distribution ---
diagnosis
B    357
M    212
Name: count, dtype: int64

--- After Encoding Target ---
   diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0          1        17.99         10.38          122.80     1001.0   
1          1        20.57         17.77          132.90     1326.0   
2          1        19.69         21.25          130.00     1203.0   
3          1        11.42         20.38           77.58      386.1   
4          1        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

  

In [7]:
# X gets all columns *except* 'diagnosis'
X = df.drop('diagnosis', axis=1)

# y gets *only* the 'diagnosis' column
y = df['diagnosis']

print("\nShape of X (features):", X.shape)
print("Shape of y (target):", y.shape)


Shape of X (features): (569, 30)
Shape of y (target): (569,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 455
Testing samples: 114


In [10]:
# Initialize the scaler
scaler = StandardScaler()

# 1. Fit the scaler on the *training data only* to learn its mean and standard deviation
# 2. Transform the training data
X_train_scaled = scaler.fit_transform(X_train)

# 3. Transform the *test data* using the scaler we *already fit* to the training data
X_test_scaled = scaler.transform(X_test)

In [11]:
print("--- Starting SVM ---")

# 1. Define the parameters we want to test
svm_params = {
    'C': [0.1, 1, 10, 100],        # Regularization strength
    'kernel': ['linear', 'rbf']     # Type of SVM
}

# 2. Set up GridSearchCV
# This will test all combinations (4 * 2 = 8) of parameters
# cv=5 means 5-fold cross-validation
# scoring='recall' means we want to find the model with the best *recall*
svm_grid = GridSearchCV(SVC(probability=True), svm_params, cv=5, scoring='recall', verbose=1)

# 3. Train the model! (This might take a moment)
svm_grid.fit(X_train_scaled, y_train)

# 4. Get the best model and its score
print("\nBest SVM Parameters:", svm_grid.best_params_)
svm_best = svm_grid.best_estimator_

# 5. Make predictions on the *scaled test data*
y_pred_svm = svm_best.predict(X_test_scaled)

# 6. Show the results
print("\n--- SVM Classification Report ---")
# '1' is Malignant (which we encoded from 'M')
print(classification_report(y_test, y_pred_svm, target_names=['Benign (0)', 'Malignant (1)']))

--- Starting SVM ---
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best SVM Parameters: {'C': 10, 'kernel': 'rbf'}

--- SVM Classification Report ---
               precision    recall  f1-score   support

   Benign (0)       0.97      0.99      0.98        71
Malignant (1)       0.98      0.95      0.96        43

     accuracy                           0.97       114
    macro avg       0.97      0.97      0.97       114
 weighted avg       0.97      0.97      0.97       114



In [12]:
print("\n--- Starting Random Forest ---")

# 1. Define parameters to test
rf_params = {
    'n_estimators': [50, 100, 200],   # Number of trees
    'max_depth': [None, 10, 20]      # How deep each tree can grow
}

# 2. Set up GridSearchCV
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='recall', verbose=1)

# 3. Train the model
# Note: Random Forest doesn't *need* scaled data, but it's fine to use it.
rf_grid.fit(X_train_scaled, y_train)

# 4. Get the best model
print("\nBest Random Forest Parameters:", rf_grid.best_params_)
rf_best = rf_grid.best_estimator_

# 5. Make predictions
y_pred_rf = rf_best.predict(X_test_scaled)

# 6. Show the results
print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred_rf, target_names=['Benign (0)', 'Malignant (1)']))


--- Starting Random Forest ---
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Best Random Forest Parameters: {'max_depth': None, 'n_estimators': 200}

--- Random Forest Classification Report ---
               precision    recall  f1-score   support

   Benign (0)       0.96      0.99      0.97        71
Malignant (1)       0.98      0.93      0.95        43

     accuracy                           0.96       114
    macro avg       0.97      0.96      0.96       114
 weighted avg       0.97      0.96      0.96       114



In [13]:
# We have a clear winner! Let's save the SVM model and the scaler.

# 1. Save the best SVM model
joblib.dump(svm_best, 'breast_cancer_model.pkl')

# 2. Save the scaler (this is the same scaler as before)
joblib.dump(scaler, 'scaler.pkl')

print("Best model (SVM) and Scaler have been saved!")

Best model (SVM) and Scaler have been saved!


In [8]:
print("New shape of X (features):", X.shape)

New shape of X (features): (569, 30)
